# Trabajo del grupo 6, datos de las 5 grandes ligas europeas del fútbol
Alejandro Álvarez Segoviano e Ivo Reguero Ferrer

# 0. Importamos librerías

In [155]:
import pandas as pd
from pathlib import Path

In [156]:
ruta = Path().resolve().parent
ruta = ruta / "data" / "fulbo.csv"
ruta

PosixPath('/data/fulbo.csv')

# 1. Primer vistazo a los datos y EDA

## 1.1 Mirada inicial

Vamos a comenzar leyendo los datos, observando de cuántas observaciones disponemos, cuántos atributos y de qué naturaleza, previo a la modelización para entender mejor la naturaleza de ellas, cómo se relacionan y si presentan limitaciones o sesgos, para hallar de que tipo son.

In [157]:
df = pd.read_csv(ruta,header=1) # para no agrupar por el tipo de variable

In [158]:
df.head()

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,PKatt,CrdY,CrdR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK,Matches,-9999
0,1,Brenden Aaronson,us USA,MFFW,Leeds United,eng Premier League,24.0,2000.0,37,30,...,0,3,0,0.15,0.18,0.33,0.15,0.33,Matches,5bc43860
1,2,Jerome Abbey,eng ENG,MF,Wolves,eng Premier League,15.0,2009.0,1,0,...,0,0,0,0.00,0.00,0.00,0.00,0.00,Matches,4f504fcc
2,3,Zach Abbott,eng ENG,DF,Nottingham,eng Premier League,19.0,2006.0,3,2,...,0,0,0,0.00,0.00,0.00,0.00,0.00,Matches,c103921a
3,4,Jones El-Abdellaoui,ma MAR,MFFW,Celta Vigo,es La Liga,19.0,2006.0,22,4,...,0,0,0,0.26,0.00,0.26,0.26,0.26,Matches,29dd3b2e
4,5,Himad Abdelli,dz ALG,MF,Marseille,fr Ligue 1,25.0,1999.0,8,1,...,0,1,0,0.00,0.00,0.00,0.00,0.00,Matches,7978cbf0


In [159]:
print(f"Disponemos de {df.shape[0]} jugadores y {df.shape[1]} atributos.")

Disponemos de 2839 jugadores y 27 atributos.


Esos 27 atributos son los siguientes :

In [160]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2839 entries, 0 to 2838
Data columns (total 27 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Rk       2839 non-null   int64  
 1   Player   2839 non-null   object 
 2   Nation   2836 non-null   object 
 3   Pos      2839 non-null   object 
 4   Squad    2839 non-null   object 
 5   Comp     2839 non-null   object 
 6   Age      2837 non-null   float64
 7   Born     2837 non-null   float64
 8   MP       2839 non-null   int64  
 9   Starts   2839 non-null   int64  
 10  Min      2839 non-null   int64  
 11  90s      2839 non-null   float64
 12  Gls      2839 non-null   int64  
 13  Ast      2839 non-null   int64  
 14  G+A      2839 non-null   int64  
 15  G-PK     2839 non-null   int64  
 16  PK       2839 non-null   int64  
 17  PKatt    2839 non-null   int64  
 18  CrdY     2839 non-null   int64  
 19  CrdR     2839 non-null   int64  
 20  Gls.1    2839 non-null   float64
 21  Ast.1    2839 

Algunos atributos tienen nulos, de igual forma cada variable individualmente será revisada después por si hay nulos codificados.

La página web de la que provienen los datos posee un glosario que usaremos para poder explicar qué significa cada dato, en la siguiente sección lo usaremos.

## 1.2 Explicación de cada atributo

*Rk*: Entero, basura que se ha quedado tras haber obtenido el CSV de la página web. No es más que un índice. Lo descartaremos después.

Tal y como se indica en la página web, y como se muestra en el csv antes de ser procesado, hay cuatro grupos de columnas:
- (sin nombre oficial) Datos de "fuera del campo" del jugador (a la que pertenecen _Player_, _Nation_, _Pos_, _Squad_, _Comp_, _Age_ y _Born_)
- _Playing Time_ (tiempo de juego, pertenecen _MP_, _Starts_, _Min_ y _90s_)
- _Performance_ (rendimiento, pertenecen _Gls_, _Ast_, _G+A_, _G-PK_, _PK_, _PKatt_, _CrdY_ y _CrdR_)
- _Per 90 Minutes_ (valores divididos entre 90 minutos, pertenecen _Gls.1_, _Ast.1_, _G+A.1_, _G-PK.1_ y _G+A-PK_)

Sin siquiera haber empezado a explicar qué significa cada atributo, vemos que muchos son fabricados a partir de otros (muchos son simples divisiones de algún atributo entre _Min_).

_Player_: El nombre del jugador, string. Puede haber repetidos, ya que un mismo jugador puede haber jugado en la misma temporada en dos clubs distintos (hay dos Himad Abdelli con la misma edad y club distinto, lo más probable es que sea la misma persona que ha cambiado de club durante el mercado de fichajes de invierno), y dos jugadores distintos pueden tener el mismo nombre (hay dos David López con una edad muy distinta)

Después de mirar más, descubrimos que cuatro jugadores son homónimos. Por suerte, ningún par de jugadores pertenece al mismo club, así que podemos diferenciar todos los jugadores con -9999. Algunos han cambiado de club, pero como vamos a tratar a cada fila como un jugador distinto, esto no importa mucho.


_Nation_: Nacionalidad del jugador, string. La nacionalidad es escogida comprobando para cuál selección han jugado, y si no han jugado para ninguna se escoge dependiendo de la nacionalidad indicada en Wikipedia o su lugar de nacimiento si está disponible.

Un jugador, Nathan Mbala, no tiene asociada una nacionalidad. Lo más probable es que esta variable no va a ser usada en ningún modelo entonces no importa mucho.

_Pos_: la posición en la que juega el jugador, categórico y **nuestra variable a predecir** (no del todo, luego definimos bien). Puede ser una o más de estas, las cuales no están separadas por espacios:

- GK - Portero
- DF - Defensa
- MF - Mediocentro
- FW - Delantero

Hemos borrado de esta lista, sacada del glosario, los roles que no aparecen representados en la base de datos. En este código se puede ver que solamente aparecen esas 4 posiciones y algunas combinaciones:

In [161]:
df["Pos"].value_counts()

,count
Pos,
MF,923
DF,705
FW,395
MFFW,217
GK,194
FWMF,146
DFMF,141
MFDF,115
DFFW,3


_Squad_: El equipo para el que han jugado, string. Puede no ser toda la temporada, como se menciona en la sección de _Player_.

_Comp_: la competición a la que pertenece el equipo, string. Son 5, como era de esperar:

In [162]:
df["Comp"].value_counts()

,count
Comp,
es La Liga,611
it Serie A,609
fr Ligue 1,561
eng Premier League,551
de Bundesliga,507


_Age_: Edad del jugador. Entero.

_Born_: Año de nacimiento del jugador, entero. Vamos a ver rápidamente si los dos atributos son calculados el uno con el otro:

In [163]:
df["Age"].value_counts()

,count
Age,
22.0,244
25.0,231
24.0,224
27.0,218
23.0,211
28.0,192
26.0,189
20.0,186
21.0,176


In [164]:
df["Born"].value_counts()

,count
Born,
2002.0,234
2003.0,230
2000.0,227
2001.0,214
1999.0,207
1997.0,204
1998.0,196
2004.0,169
2005.0,165


Al haber 6 jugadores que tienen 15 años y un solo jugador nacido en 2010, parece que la edad no es calculada con 2025 - _Born_. Eso sí, puede que tengamos que borrar uno de los dos en un futuro.

_MP_: Partidos jugados, entero. Entre 1 y 38, en el caso de la liga alemana y la francesa el máximo es menos, hay menos equipos en esas ligas y menos partidos (máximo de 34).

_Starts_: Partidos en los que el jugador fue titular, entero. Mismo rango que el atributo anterior.

_Min_: Minutos jugados a lo largo de la temporada, float.

_90s_ : Minutos jugados dividido por 90, float. Parece que no se cuentan los minutos jugados en el tiempo añadido, ya que esta métrica tiene como máximo 38.0.

*Gls*: Goles, en total. Entero.

*Ast*: Asistencias, en total. Entero.

*G+A*: Suma de las dos anteriores variables. Entero.

*G-PK*: Goles menos goles marcados de penalti. Entero.

*PK*: Goles de penalti. Entero.

*PKatt*: Penaltis tirados. Entero.

*CrdY* y *CrdR*: Tarjetas amarillas y rojas, enteros. Normalmente, con 5 amarillas acumuladas te pierdes el siguiente partido y te cambian durante el partido en el que te la ponen, para evitar que te pongan roja. La roja puede causar que te pierdas varios partidos, además de que te impide seguir jugando el partido en el que te la ponen. Además, los defensas suelen ser amonestados más, cosa que nos podría servir.

_Per 90 Minutes_ (valores divididos entre 90 minutos, pertenecen _Gls.1_, _Ast.1_, _G+A.1_, _G-PK.1_ y _G+A-PK_): Son valores como los anteriores, pero divididos entre 90, floats. El único que no tiene un equivalente a los anteriores es *G+A-PK*, el cual también es calculable con los datos anteriores. Lo más probable es que en cuanto hagamos selección de variables u otro método que reduzca las dimensiones estas variables o las anteriores terminen desapareciendo, puesto que sin siquiera mirarlo su correlación de Pearson con algunas de las variables anteriores será (al menos casi) 1.


*Matches* y *-9999*: Ambos, en la página web, nos indicaban identificación de un jugador. Matches se vuelve inútil (en la web es un hiperlink que te lleva a los partidos jugados por el jugador, aquí es Matches ad nauseum) y -9999 identifica correctamente jugadores distintos.

##1.3 Borrando casos especiales y excluyendo variables

A partir de este punto, sabemos un poco lo que queremos hacer con nuestra base de datos (Diferenciar delanteros del resto) ya que las únicas estadísticas de partido que tenemos son los goles y asistencias, estadísticas ofensivas que no permiten diferenciar a jugadores defensivos de porteros, por ejemplo.

Vamos a querer evitar que aparezcan dos tipos de outliers, los porteros y los jugadores de cantera. Los porteros, exceptuando unos pocos casos, no marcan, y los jugadores de cantera (y los jugadores muy revulsivos) apenas juegan. El problema con los primeros es que no son jugadores de campo, así que no van a tener apenas estadísticas ofensivas y van a ser muy fácilmente clasificables, y el de los segundos es que juegan poco, y eso puede causar outliers extremos. Ejemplo, un canterano que marca un gol en un partido y no vuelve a jugar podría tener un goles por 90 de 3, lo cual es un outlier extremo.

Vamos a borrar las filas que tienen pos = GK y las que han jugado menos de 90 minutos:

In [165]:
df_filtrado = df[(df['Pos'] != 'GK') & (df['Min'] >= 90)]
print(f"Quedan {df_filtrado.shape[0]} jugadores después del filtrado.")

Quedan 2290 jugadores después del filtrado.


Ahora, hay tres valores nulos en toda la base de datos: la nacionalidad de un jugador y la edad/año de nacimiento de otros dos. Vamos a descartarlos, dado que son una enana parte de la muestra:

In [166]:
df_filtrado = df_filtrado.dropna()
print(f"Quedan {df_filtrado.shape[0]} jugadores tras eliminar los valores nulos.")

Quedan 2287 jugadores tras eliminar los valores nulos.


Lo mejor sería tratar a cada fila como si fuese un jugador distinto. ¿Por qué? Porque cambiar de equipo puede cambiar tu posición habitual, tu liga (en la Bundesliga se marca muchas más veces por partido, por ejemplo) y tu estado de ánimo (un jugador descontento cambia de club y empieza a rendir en su nuevo club). No queremos, entonces, conocer al jugador y vamos a borrar todo lo que lo identifique.

Tras haber sido transformado a un CSV, hemos perdido información a la que podríamos acceder desde la página web: enlaces en el nombre de los jugadores que nos permitían acceder a un perfil de la página del jugador con más información, lo cual ocurre también con la selección, el equipo y la liga, y la razón por la que se ha escogido esa nacionalidad para el jugador.

Otro dato que perdemos es el enlace que te conecta a los partidos que ha jugado en esa temporada el jugador. Si quisiésemos recuperar el enlace, en el caso de Brenden Aaronson, tendríamos que reconstruirlo usando el atributo de la última columna, la temporada que queremos mirar (en nuestro caso 2025-2026) y su nombre separado por guiones para poder ver todos los partidos que jugó durante la temporada. (https://fbref.com/en/players/5bc43860/matchlogs/2025-2026/summary/Brenden-Aaronson-Match-Logs)

Ahora que sabemos que estas columnas no nos sirven para modelar, podemos borrar las columnas _Matches_ y _-9999_.

Un dato más que no nos sirve es _Rk_, un dato usado dentro de la página web para ver el la posición del jugador cuando estás ordenando dependiendo de un dato (si ordenas por más partidos jugados, _Rk_ se actualiza en la página web). Es decir, es un índice, no lo queremos.

**Vamos a borrar esas tres columnas:**

In [167]:
df_filtrado = df_filtrado.drop(columns=['Rk', 'Matches', '-9999'])
df_filtrado.head()

,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,Min,...,G-PK,PK,PKatt,CrdY,CrdR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK
0,Brenden Aaronson,us USA,MFFW,Leeds United,eng Premier League,24.0,2000.0,37,30,2449,...,4,0,0,3,0,0.15,0.18,0.33,0.15,0.33
2,Zach Abbott,eng ENG,DF,Nottingham,eng Premier League,19.0,2006.0,3,2,164,...,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00
3,Jones El-Abdellaoui,ma MAR,MFFW,Celta Vigo,es La Liga,19.0,2006.0,22,4,681,...,2,0,0,0,0,0.26,0.00,0.26,0.26,0.26
4,Himad Abdelli,dz ALG,MF,Marseille,fr Ligue 1,25.0,1999.0,8,1,138,...,0,0,0,1,0,0.00,0.00,0.00,0.00,0.00
5,Himad Abdelli,dz ALG,MF,Angers,fr Ligue 1,25.0,1999.0,13,11,943,...,0,2,2,1,0,0.19,0.00,0.19,0.00,0.00


Más cosas que no queremos son el nombre, la nación y el equipo. El nombre no lo queremos porque identifica al jugador, y la nación y el equipo nos forzaría a añadir muchísimas variables dummy (~110 naciones y 96 equipos distintos). No queremos sufrir de maldición de la dimensionalidad así que vamos a excluirlas también.

**Borremos otras tres columnas:**

In [168]:
df_filtrado = df_filtrado.drop(columns=['Player', 'Nation', 'Squad'])
df_filtrado.head()

,Pos,Comp,Age,Born,MP,Starts,Min,90s,Gls,Ast,...,G-PK,PK,PKatt,CrdY,CrdR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK
0,MFFW,eng Premier League,24.0,2000.0,37,30,2449,27.2,4,5,...,4,0,0,3,0,0.15,0.18,0.33,0.15,0.33
2,DF,eng Premier League,19.0,2006.0,3,2,164,1.8,0,0,...,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00
3,MFFW,es La Liga,19.0,2006.0,22,4,681,7.6,2,0,...,2,0,0,0,0,0.26,0.00,0.26,0.26,0.26
4,MF,fr Ligue 1,25.0,1999.0,8,1,138,1.5,0,0,...,0,0,0,1,0,0.00,0.00,0.00,0.00,0.00
5,MF,fr Ligue 1,25.0,1999.0,13,11,943,10.5,2,0,...,0,2,2,1,0,0.19,0.00,0.19,0.00,0.00


Cambiemos el formato con el que guardamos la *Pos* y la *Comp*:

En el caso de *Comp*, vamos a hacer *one hot encoding* porque queremos saber a que liga pertenece cada jugador, y queremos guardarlo en algo que no sea un string.

En el de *Pos* lo que haremos es convertirla en una binaria, queremos saber si el jugador juega de delantero (tiene FW o alguna combinación que lo incluya) o no. Arriba hemos visto que los que contienen *foward* son FW, MFFW, FWMF y DFFW, serán 1. El resto serán 0.

Hagamos las transformaciones:

In [169]:
df_filtrado = pd.get_dummies(df_filtrado, columns=['Comp'], dtype=int, drop_first=True) # perdemos bundesliga
# pero si no perteneces a ninguna de las cuatro ligas representadas por el dummy, eres de la bundesliga
# no pierdes info ni eres redundante asi
df_filtrado.head()

,Pos,Age,Born,MP,Starts,Min,90s,Gls,Ast,G+A,...,CrdR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK,Comp_eng Premier League,Comp_es La Liga,Comp_fr Ligue 1,Comp_it Serie A
0,MFFW,24.0,2000.0,37,30,2449,27.2,4,5,9,...,0,0.15,0.18,0.33,0.15,0.33,1,0,0,0
2,DF,19.0,2006.0,3,2,164,1.8,0,0,0,...,0,0.00,0.00,0.00,0.00,0.00,1,0,0,0
3,MFFW,19.0,2006.0,22,4,681,7.6,2,0,2,...,0,0.26,0.00,0.26,0.26,0.26,0,1,0,0
4,MF,25.0,1999.0,8,1,138,1.5,0,0,0,...,0,0.00,0.00,0.00,0.00,0.00,0,0,1,0
5,MF,25.0,1999.0,13,11,943,10.5,2,0,2,...,0,0.19,0.00,0.19,0.00,0.00,0,0,1,0


In [170]:
df_filtrado['EsDelantero'] = df_filtrado['Pos'].str.contains('FW').astype(int)
display(df_filtrado[['Pos', 'EsDelantero']].value_counts()) # comprobando si estan bien asignados
# los tres delantero-defensas han sido excluidos por ser canteranos o por valores faltantes, una de dos
print(df_filtrado['EsDelantero'].value_counts()) # veamos cuantos tenemos de cada tipo

,,count
Pos,EsDelantero,
MF,0,787
DF,0,625
FW,1,294
MFFW,1,216
DFMF,0,129
FWMF,1,121
MFDF,0,115


EsDelantero
0    1656
1     631
Name: count, dtype: int64


In [171]:
# ahora que hemos visto que esta bien, borramos la variable pos
df_filtrado = df_filtrado.drop(columns=['Pos'])

Por último, tenemos que pensar cuáles de las variables redundantes queremos borrar. Quitaremos cuatro selectas: *Born*, *90s*, *G+A* y *G+A.1*, porque son combinaciones lineales exactas. Otras, que se van a conservar, también lo son, pero las conservamos para ver cuál preferimos después de hacer otro filtrado o una regularización.

In [172]:
df_filtrado = df_filtrado.drop(columns=['Born', '90s', 'G+A', 'G+A.1'])

Nuestra base de datos termina quedando así, después de todo esto:

In [173]:
df_filtrado.head()

,Age,MP,Starts,Min,Gls,Ast,G-PK,PK,PKatt,CrdY,CrdR,Gls.1,Ast.1,G-PK.1,G+A-PK,Comp_eng Premier League,Comp_es La Liga,Comp_fr Ligue 1,Comp_it Serie A,EsDelantero
0,24.0,37,30,2449,4,5,4,0,0,3,0,0.15,0.18,0.15,0.33,1,0,0,0,1
2,19.0,3,2,164,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,1,0,0,0,0
3,19.0,22,4,681,2,0,2,0,0,0,0,0.26,0.00,0.26,0.26,0,1,0,0,1
4,25.0,8,1,138,0,0,0,0,0,1,0,0.00,0.00,0.00,0.00,0,0,1,0,0
5,25.0,13,11,943,2,0,0,2,2,1,0,0.19,0.00,0.00,0.00,0,0,1,0,0


In [175]:
print(f"Disponemos de {df_filtrado.shape[0]} jugadores y {df_filtrado.shape[1]} atributos.")

Disponemos de 2287 jugadores y 20 atributos.


## 1.5 EDA

---

#2 cosas

-------------------------------------------------------------------------------------------------------------------------------

to do, tareas

doble analisis exploratorio para buscar un objetivo?? scatterplot?